# MIMIC-IV Dx Prediction (Prior Admissions → Next Admission) — Improved + Clustering Notebook

This version extends the improved notebook with:

- **Sparse/dense split optimizers** so the `EmbeddingBag` model trains correctly
- **Fast threshold tuning** (optional)
- **Patient clustering** on learned fused embeddings
- Cluster profiling by diagnosis burden, demographics, and dominant diagnoses

The main modeling lens remains **Top-K retrieval**, and clustering is used to discover patient subgroups from the learned representation.


In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

pd.set_option("display.max_columns", 200)

from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score


## 0) Point to your MIMIC-IV CSV folder

Set `DATA_PATH` to the folder containing the MIMIC-IV tables you already used (the same as your old notebook).

Expected files (at least):  
- `admissions.csv`, `patients.csv`, `diagnoses_icd.csv`, `d_icd_diagnoses.csv`


In [ ]:
DATA_PATH = os.environ.get("MIMIC_DATA_PATH", "/content/drive/MyDrive/capstone_prescription_data/data")
assert os.path.exists(DATA_PATH), f"DATA_PATH does not exist: {DATA_PATH}"
print("DATA_PATH:", DATA_PATH)

# Load CSVs into a dict by filename (without extension)
datasets = {}
for file in os.listdir(DATA_PATH):
    if file.endswith(".csv"):
        key = file.replace(".csv","")
        datasets[key] = pd.read_csv(os.path.join(DATA_PATH, file))
print("Loaded:", sorted(datasets.keys()))


## 1) Build `grouped_data` (admission-level with diagnosis list)

This mirrors your existing pipeline, excluding prescriptions.


In [ ]:
required = ["admissions", "patients", "diagnoses_icd", "d_icd_diagnoses"]
missing = [k for k in required if k not in datasets]
assert not missing, f"Missing required tables: {missing}"

admissions = datasets["admissions"].copy()
patients = datasets["patients"].copy()
diagnoses = datasets["diagnoses_icd"].copy()
d_dx = datasets["d_icd_diagnoses"].copy()

# Join diagnoses with titles
dx = diagnoses.merge(
    d_dx[["icd_code","icd_version","long_title"]],
    on=["icd_code","icd_version"],
    how="left"
)

# Basic admission-level join
adm = admissions.merge(patients, on="subject_id", how="left")

# Aggregate diagnoses per (subject_id, hadm_id)
dx_agg = (
    dx.groupby(["subject_id","hadm_id"], observed=True)["long_title"]
      .apply(list)
      .reset_index()
)

grouped_data = adm.merge(dx_agg, on=["subject_id","hadm_id"], how="left")
grouped_data["long_title"] = grouped_data["long_title"].apply(lambda x: x if isinstance(x, list) else [])
print("grouped_data shape:", grouped_data.shape)
print("subjects:", grouped_data["subject_id"].nunique())
print("unique diagnosis titles:", len(pd.Index(dx["long_title"].dropna().unique())))


## 2) Feature builder (same logic as your notebook)

We keep your strong baseline feature engineering, but later we upgrade:
- **Sparse dx history** stays sparse end-to-end (EmbeddingBag)
- **Inference** uses Top‑K / tuned threshold


In [ ]:
def ensure_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    return [x]

def mode_per_group_counts(df, col):
    # Fast mode per subject_id using value_counts
    vc = df.groupby("subject_id")[col].value_counts(dropna=False)
    # take first (most frequent) per group
    return vc.groupby(level=0).idxmax().apply(lambda t: t[1])

def build_features_and_labels(grouped_data: pd.DataFrame):
    df = grouped_data.copy()

    df["admittime"] = pd.to_datetime(df["admittime"])
    df["dischtime"] = pd.to_datetime(df["dischtime"])
    df["number_of_days_stay"] = (df["dischtime"] - df["admittime"]).dt.total_seconds() / 86400.0

    df["long_title"] = df["long_title"].apply(ensure_list)

    # target admission = most recent per subject
    idx_target = df.groupby("subject_id")["admittime"].idxmax()
    target_df = df.loc[idx_target].copy()
    prior_df = df.drop(idx_target).copy()

    # base aggregates from priors (subjects with >=1 prior)
    base = (
        prior_df.groupby("subject_id")
        .agg(
            number_of_prior_admissions=("hadm_id", "nunique"),
            number_of_total_days_stay=("number_of_days_stay", "sum"),
            average_days_per_admission=("number_of_days_stay", "mean"),
            median_days_per_admission=("number_of_days_stay", "median"),
            max_days_per_admission=("number_of_days_stay", "max"),
            min_days_per_admission=("number_of_days_stay", "min"),
            standard_deviation_days_per_admission=("number_of_days_stay", "std"),
        )
    )

    # recent prior admission per subject
    idx_prior_recent = prior_df.groupby("subject_id")["admittime"].idxmax()
    prior_recent = prior_df.loc[idx_prior_recent].set_index("subject_id")

    base["recent_admission_type"] = prior_recent["admission_type"]
    base["recent_admit_provider_id"] = prior_recent["admit_provider_id"]
    base["recent_admission_location"] = prior_recent["admission_location"]
    base["recent_discharge_location"] = prior_recent["discharge_location"] if "discharge_location" in prior_recent.columns else "Unknown"
    base["recent_insurance"] = prior_recent["insurance"]
    base["recent_language"] = prior_recent["language"]
    base["recent_marital_status"] = prior_recent["marital_status"]
    base["recent_number_of_days_stay"] = prior_recent["number_of_days_stay"]
    base["most_recent_race"] = prior_recent["race"]
    base["most_recent_gender"] = prior_recent["gender"]
    base["age"] = prior_recent["anchor_age"]

    # frequent/mode from priors
    base["frequent_admission_type"] = mode_per_group_counts(prior_df, "admission_type")
    base["frequent_admit_provider_id"] = mode_per_group_counts(prior_df, "admit_provider_id")
    base["frequent_admission_location"] = mode_per_group_counts(prior_df, "admission_location")
    if "discharge_location" in prior_df.columns:
        base["frequent_discharge_location"] = mode_per_group_counts(prior_df, "discharge_location")
    base["frequent_insurance"] = mode_per_group_counts(prior_df, "insurance")

    # attach target metadata (optional; remove if you consider leakage)
    target_df = target_df.set_index("subject_id")
    base["target_admission_type"] = target_df["admission_type"]
    base["target_admit_provider_id"] = target_df["admit_provider_id"]
    base["target_admission_location"] = target_df["admission_location"]
    base["target_insurance"] = target_df["insurance"]
    base["target_language"] = target_df["language"]
    base["target_marital_status"] = target_df["marital_status"]
    base["target_race"] = target_df["race"]
    base["target_gender"] = target_df["gender"]
    base["target_age"] = target_df["anchor_age"]

    base = base.reset_index()

    # ---- Build sparse X_diag (prior counts) and y (target multi-hot) ----
    prior_dx = prior_df[["subject_id", "long_title"]].explode("long_title").dropna(subset=["long_title"])
    prior_counts = (
        prior_dx.groupby(["subject_id", "long_title"], observed=True)
        .size()
        .rename("cnt")
        .reset_index()
    )

    target_dx = target_df.reset_index()[["subject_id", "long_title"]].explode("long_title").dropna(subset=["long_title"])
    target_dx = target_dx.drop_duplicates(["subject_id", "long_title"])

    diag_vocab = pd.Index(
        pd.concat([prior_counts["long_title"], target_dx["long_title"]], ignore_index=True).unique()
    )
    diag_to_idx = {d: i for i, d in enumerate(diag_vocab)}
    L = len(diag_vocab)
    print(f"Diagnosis vocab size = {L}")

    base_subjects = base["subject_id"].values
    target_subjects = target_dx["subject_id"].unique()
    common = np.intersect1d(base_subjects, target_subjects)

    base = base[base["subject_id"].isin(common)].reset_index(drop=True)
    subj_order = base["subject_id"].values
    subj_to_row = {sid: i for i, sid in enumerate(subj_order)}
    N = len(subj_order)
    print(f"Subjects kept (>=1 prior + has target) = {N}")

    prior_counts = prior_counts[prior_counts["subject_id"].isin(common)]
    row_idx = prior_counts["subject_id"].map(subj_to_row).to_numpy()
    col_idx = prior_counts["long_title"].map(diag_to_idx).to_numpy()
    data = prior_counts["cnt"].astype(np.float32).to_numpy()
    X_diag_csr = sp.csr_matrix((data, (row_idx, col_idx)), shape=(N, L), dtype=np.float32)

    target_dx = target_dx[target_dx["subject_id"].isin(common)]
    y_row = target_dx["subject_id"].map(subj_to_row).to_numpy()
    y_col = target_dx["long_title"].map(diag_to_idx).to_numpy()
    y_data = np.ones(len(target_dx), dtype=np.float32)
    y_csr = sp.csr_matrix((y_data, (y_row, y_col)), shape=(N, L), dtype=np.float32)

    return base, X_diag_csr, y_csr, diag_vocab


In [ ]:
base_df, X_diag_csr, y_csr, diag_vocab = build_features_and_labels(grouped_data)
print("base_df:", base_df.shape, "X_diag:", X_diag_csr.shape, "y:", y_csr.shape)


## 3) Preprocess tabular features (OneHot + StandardScaler)

We keep tabular features as **sparse** too (it scales and works well).


In [ ]:
X_tab = base_df.drop(columns=["subject_id"]).copy()

cat_cols = [c for c in X_tab.columns if X_tab[c].dtype == "object"]
num_cols = [c for c in X_tab.columns if c not in cat_cols]

# Fill missing
for c in num_cols:
    X_tab[c] = X_tab[c].fillna(X_tab[c].median())
for c in cat_cols:
    X_tab[c] = X_tab[c].fillna("Unknown").astype(str)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("scaler", StandardScaler(with_mean=False))]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse=True), cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

# Split by row index (already per-subject)
idx = np.arange(len(X_tab))
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=42)
train_idx, val_idx = train_test_split(train_idx, test_size=0.1, random_state=42)

X_train_tab = preprocess.fit_transform(X_tab.iloc[train_idx])
X_val_tab   = preprocess.transform(X_tab.iloc[val_idx])
X_test_tab  = preprocess.transform(X_tab.iloc[test_idx])

X_train_diag = X_diag_csr[train_idx]
X_val_diag   = X_diag_csr[val_idx]
X_test_diag  = X_diag_csr[test_idx]

y_train = y_csr[train_idx]
y_val   = y_csr[val_idx]
y_test  = y_csr[test_idx]

print("Tab shapes (sparse):", X_train_tab.shape, X_val_tab.shape, X_test_tab.shape)
print("Diag shapes (csr):", X_train_diag.shape, X_val_diag.shape, X_test_diag.shape)


## 4) Torch Dataset for sparse diag history (EmbeddingBag-ready)

We avoid `toarray()` for diagnosis history.  
Each row provides:
- tabular sparse row (we densify **after** preprocessing; dimensionality is manageable)
- diagnosis indices + counts (sparse)
- target indices (sparse)


In [ ]:
def csr_row_indices_values(csr_mat, i):
    start = csr_mat.indptr[i]
    end = csr_mat.indptr[i+1]
    idx = csr_mat.indices[start:end]
    val = csr_mat.data[start:end]
    return idx.astype(np.int64), val.astype(np.float32)

class DxDataset(Dataset):
    def __init__(self, X_tab_csr, X_diag_csr, y_csr):
        self.X_tab = X_tab_csr.tocsr()
        self.X_diag = X_diag_csr.tocsr()
        self.y = y_csr.tocsr()

    def __len__(self):
        return self.X_tab.shape[0]

    def __getitem__(self, i):
        # tabular row to dense float32
        x_tab = self.X_tab.getrow(i).toarray().astype(np.float32).ravel()

        d_idx, d_val = csr_row_indices_values(self.X_diag, i)
        y_idx, _ = csr_row_indices_values(self.y, i)

        return x_tab, d_idx, d_val, y_idx

def collate_dx(batch, L):
    # batch: list of (x_tab, d_idx, d_val, y_idx)
    x_tabs = torch.tensor(np.stack([b[0] for b in batch], axis=0), dtype=torch.float32)

    # EmbeddingBag format: concatenate indices; offsets mark start of each sample
    all_idx = []
    all_val = []
    offsets = [0]
    for (_, d_idx, d_val, _) in batch:
        all_idx.append(torch.tensor(d_idx, dtype=torch.int64))
        all_val.append(torch.tensor(d_val, dtype=torch.float32))
        offsets.append(offsets[-1] + len(d_idx))
    all_idx = torch.cat(all_idx) if len(all_idx) else torch.empty((0,), dtype=torch.int64)
    all_val = torch.cat(all_val) if len(all_val) else torch.empty((0,), dtype=torch.float32)
    offsets = torch.tensor(offsets[:-1], dtype=torch.int64)  # len = batch_size

    # Multi-hot targets as dense (per-batch only)
    y = torch.zeros((len(batch), L), dtype=torch.float32)
    for i, (_, _, _, y_idx) in enumerate(batch):
        if len(y_idx):
            y[i, torch.tensor(y_idx, dtype=torch.int64)] = 1.0

    return x_tabs, all_idx, offsets, all_val, y


## 5) More advanced model: Dx EmbeddingBag + Tabular tower + Gated Fusion

**Why it’s better than the current dense “diag vector → MLP”:**
- Diagnosis history is **sparse** and high-cardinality → EmbeddingBag is the standard scalable solution.
- Fusion layer learns when to trust tabular vs dx history.

Architecture:
- Tabular: MLP
- Dx: EmbeddingBag (weighted by counts) → LayerNorm → MLP
- Fusion: gated cross + residual MLP


In [ ]:
class GatedFusion(nn.Module):
    def __init__(self, a_dim, b_dim, out_dim):
        super().__init__()
        self.proj_a = nn.Linear(a_dim, out_dim)
        self.proj_b = nn.Linear(b_dim, out_dim)
        self.gate = nn.Sequential(
            nn.Linear(out_dim * 2, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
            nn.Sigmoid(),
        )

    def forward(self, a, b):
        a2 = self.proj_a(a)
        b2 = self.proj_b(b)
        g = self.gate(torch.cat([a2, b2], dim=1))
        return g * a2 + (1 - g) * b2

class DxTabNet(nn.Module):
    def __init__(self, tab_dim, num_labels, dx_vocab_size,
                 dx_emb_dim=128, tab_hidden=(512,256), dx_hidden=(256,),
                 fusion_dim=256, dropout=0.2):
        super().__init__()
        self.dx_emb = nn.EmbeddingBag(dx_vocab_size, dx_emb_dim, mode="sum", sparse=True)
        self.dx_ln = nn.LayerNorm(dx_emb_dim)

        def mlp(in_dim, hidden):
            layers=[]
            d=in_dim
            for h in hidden:
                layers += [nn.Linear(d,h), nn.ReLU(), nn.Dropout(dropout)]
                d=h
            return nn.Sequential(*layers), d

        self.tab_mlp, tab_out = mlp(tab_dim, tab_hidden)
        self.dx_mlp, dx_out = mlp(dx_emb_dim, dx_hidden)

        self.fuse = GatedFusion(tab_out, dx_out, fusion_dim)

        self.head = nn.Sequential(
            nn.Linear(fusion_dim, fusion_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, num_labels),
        )

    def forward(self, x_tab, dx_idx, dx_offsets, dx_weights):
        # dx embeddingbag: weights apply counts
        dx_vec = self.dx_emb(dx_idx, dx_offsets, per_sample_weights=dx_weights)
        dx_vec = self.dx_ln(dx_vec)
        dx_vec = self.dx_mlp(dx_vec)

        tab_vec = self.tab_mlp(x_tab)

        fused = self.fuse(tab_vec, dx_vec)
        logits = self.head(fused)
        return logits


## 6) Loss: capped pos_weight OR focal loss (choose one)

Start with **capped pos_weight** + Top‑K decoding.  
If you still get too many false positives, try focal loss.


In [ ]:
def make_capped_pos_weight(y_csr, cap=20.0, device="cpu"):
    pos = np.asarray(y_csr.sum(axis=0)).ravel().astype(np.float64)
    N = y_csr.shape[0]
    neg = N - pos
    pw = neg / (pos + 1e-8)
    pw = np.clip(pw, 1.0, cap)
    return torch.tensor(pw, dtype=torch.float32, device=device)

class FocalBCEWithLogits(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        # logits, targets: (B, L)
        p = torch.sigmoid(logits)
        ce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p_t = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = alpha_t * (1 - p_t) ** self.gamma * ce
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


## 7) Train loop (AMP) + validation with Top‑K metrics

In [ ]:
@torch.no_grad()
def predict_proba(model, loader, device):
    model.eval()
    probs=[]
    ys=[]
    for x_tab, dx_idx, dx_offsets, dx_w, y in loader:
        x_tab = x_tab.to(device)
        dx_idx = dx_idx.to(device)
        dx_offsets = dx_offsets.to(device)
        dx_w = dx_w.to(device)
        logits = model(x_tab, dx_idx, dx_offsets, dx_w)
        probs.append(torch.sigmoid(logits).cpu().numpy())
        ys.append(y.cpu().numpy())
    return np.vstack(probs), np.vstack(ys)

def topk_pred_indices(probs, k=20):
    idx_part = np.argpartition(-probs, kth=k-1, axis=1)[:, :k]
    row = np.arange(probs.shape[0])[:, None]
    idx_sorted = idx_part[row, np.argsort(-probs[row, idx_part], axis=1)]
    return idx_sorted

def precision_recall_hits_at_k(probs, y_true_dense, k=20):
    idx = topk_pred_indices(probs, k=k)
    precisions=[]
    recalls=[]
    hits=[]
    true_counts = y_true_dense.sum(axis=1)
    for i in range(probs.shape[0]):
        pred = idx[i]
        true = y_true_dense[i].nonzero()[0]
        true_set = set(true.tolist())
        inter = sum((p in true_set) for p in pred)
        precisions.append(inter / k)
        recalls.append(inter / max(len(true_set),1))
        hits.append(1.0 if inter>0 else 0.0)
    return {
        "P@K": float(np.mean(precisions)),
        "R@K": float(np.mean(recalls)),
        "Hits@K": float(np.mean(hits)),
        "AvgTrueLabels": float(np.mean(true_counts)),
    }

def evaluate_many_k(probs, y_true_dense, ks=(5,10,20,30,50)):
    return {k: precision_recall_hits_at_k(probs, y_true_dense, k=k) for k in ks}


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

L = y_csr.shape[1]
tab_dim = X_train_tab.shape[1]

train_ds = DxDataset(X_train_tab, X_train_diag, y_train)
val_ds   = DxDataset(X_val_tab, X_val_diag, y_val)
test_ds  = DxDataset(X_test_tab, X_test_diag, y_test)

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True,
                          collate_fn=lambda b: collate_dx(b, L), num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=512, shuffle=False,
                          collate_fn=lambda b: collate_dx(b, L), num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=512, shuffle=False,
                          collate_fn=lambda b: collate_dx(b, L), num_workers=0)

model = DxTabNet(tab_dim=tab_dim, num_labels=L, dx_vocab_size=L,
                 dx_emb_dim=128, tab_hidden=(512,256), dx_hidden=(256,),
                 fusion_dim=256, dropout=0.2).to(device)

# Option 1: capped pos_weight BCE
use_focal = False
if use_focal:
    criterion = FocalBCEWithLogits(alpha=0.25, gamma=2.0).to(device)
else:
    pos_weight = make_capped_pos_weight(y_train, cap=10.0, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Sparse/dense split optimizers for EmbeddingBag
sparse_params, dense_params = [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if ("emb" in name.lower()) or ("embedding" in name.lower()):
        sparse_params.append(p)
    else:
        dense_params.append(p)

opt_dense = torch.optim.AdamW(dense_params, lr=2e-3, weight_decay=1e-4)
opt_sparse = torch.optim.SparseAdam(sparse_params, lr=2e-3) if len(sparse_params) > 0 else None
scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))

def train_one_epoch():
    model.train()
    total=0.0
    n=0
    for x_tab, dx_idx, dx_offsets, dx_w, y in train_loader:
        x_tab = x_tab.to(device, non_blocking=True)
        dx_idx = dx_idx.to(device, non_blocking=True)
        dx_offsets = dx_offsets.to(device, non_blocking=True)
        dx_w = dx_w.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        opt_dense.zero_grad(set_to_none=True)
        if opt_sparse is not None:
            opt_sparse.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device=="cuda")):
            logits = model(x_tab, dx_idx, dx_offsets, dx_w)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(opt_dense)
        torch.nn.utils.clip_grad_norm_(dense_params, 1.0)
        scaler.step(opt_dense)
        if opt_sparse is not None:
            scaler.step(opt_sparse)
        scaler.update()

        total += float(loss.detach().cpu())
        n += 1
    return total / max(n,1)

@torch.no_grad()
def validate():
    probs, y_true = predict_proba(model, val_loader, device)
    stats = evaluate_many_k(probs, y_true, ks=(5,10,20,30))
    return stats

best = None
best_p20 = -1
for epoch in range(1, 11):  # start with 10 epochs; increase if needed
    loss = train_one_epoch()
    stats = validate()
    p20 = stats[20]["P@K"]
    if p20 > best_p20:
        best_p20 = p20
        best = { "epoch": epoch, "loss": loss, "stats": stats }
        torch.save(model.state_dict(), "best_dx_tabnet.pt")
    print(f"Epoch {epoch:03d} | loss {loss:.4f} | P@20 {stats[20]['P@K']:.4f} | R@20 {stats[20]['R@K']:.4f} | Hits@20 {stats[20]['Hits@K']:.4f}")

print("Best:", best)

## 8) Final evaluation on test set (Top‑K + sanity checks)

In [ ]:
model.load_state_dict(torch.load("best_dx_tabnet.pt", map_location=device))
probs_test, y_test_dense = predict_proba(model, test_loader, device)

stats = evaluate_many_k(probs_test, y_test_dense, ks=(5,10,20,30,50))
stats


## 9) Clustering patients using learned fused embeddings

We cluster patients in the **learned representation space** rather than on raw diagnosis counts. This usually produces more meaningful subgroups because the embedding already blends diagnosis history and tabular context.

Workflow:
1. Extract fused embeddings from the trained model
2. Choose a `k` using silhouette score on a sample
3. Run `MiniBatchKMeans` on train + test embeddings
4. Profile clusters by burden and top diagnoses


In [ ]:
@torch.no_grad()
def extract_fused_embeddings(model, loader, device):
    model.eval()
    embs = []
    ys = []
    for x_tab, dx_idx, dx_offsets, dx_w, y in loader:
        x_tab = x_tab.to(device)
        dx_idx = dx_idx.to(device)
        dx_offsets = dx_offsets.to(device)
        dx_w = dx_w.to(device)

        dx_vec = model.dx_emb(dx_idx, dx_offsets, per_sample_weights=dx_w)
        dx_vec = model.dx_ln(dx_vec)
        dx_vec = model.dx_mlp(dx_vec)
        tab_vec = model.tab_mlp(x_tab)
        fused = model.fuse(tab_vec, dx_vec)

        embs.append(fused.cpu().numpy())
        ys.append(y.cpu().numpy())
    return np.vstack(embs), np.vstack(ys)

train_emb, y_train_dense = extract_fused_embeddings(model, train_loader, device)
val_emb, y_val_dense = extract_fused_embeddings(model, val_loader, device)
test_emb, y_test_dense = extract_fused_embeddings(model, test_loader, device)

all_emb = np.vstack([train_emb, val_emb, test_emb])
print("Embedding shapes:", train_emb.shape, val_emb.shape, test_emb.shape, all_emb.shape)

In [ ]:
def choose_k_by_silhouette(X, k_values=(4,6,8,10,12), sample_size=10000, random_state=42):
    rng = np.random.default_rng(random_state)
    n = X.shape[0]
    if n > sample_size:
        idx = rng.choice(n, size=sample_size, replace=False)
        X_eval = X[idx]
    else:
        X_eval = X

    rows = []
    for k in k_values:
        km = MiniBatchKMeans(n_clusters=k, random_state=random_state, batch_size=2048, n_init=10)
        labels = km.fit_predict(X_eval)
        sil = silhouette_score(X_eval, labels) if len(np.unique(labels)) > 1 else np.nan
        rows.append({"k": k, "silhouette": sil, "inertia": km.inertia_})
    return pd.DataFrame(rows)

k_search = choose_k_by_silhouette(all_emb, k_values=(4,6,8,10,12))
k_search

In [ ]:
# Pick the best k by silhouette; override manually if you want
best_k = int(k_search.sort_values("silhouette", ascending=False).iloc[0]["k"])
print("Chosen k:", best_k)

clusterer = MiniBatchKMeans(n_clusters=best_k, random_state=42, batch_size=2048, n_init=10)
all_clusters = clusterer.fit_predict(all_emb)

# map train/val/test back
n_train, n_val, n_test = len(train_emb), len(val_emb), len(test_emb)
train_clusters = all_clusters[:n_train]
val_clusters = all_clusters[n_train:n_train+n_val]
test_clusters = all_clusters[n_train+n_val:]

# attach cluster ids to the base dataframe in split order
cluster_df = base_df.iloc[np.concatenate([train_idx, val_idx, test_idx])].copy().reset_index(drop=True)
cluster_df["cluster_id"] = all_clusters
cluster_df["split"] = (["train"] * n_train) + (["val"] * n_val) + (["test"] * n_test)
cluster_df.head()

In [ ]:
# Cluster summary on the test split
true_burden_test = y_test_dense.sum(axis=1)
probs_test, _ = predict_proba(model, test_loader, device)
top20_idx = topk_pred_indices(probs_test, k=20)
pred_top20 = np.zeros_like(probs_test, dtype=np.int8)
rows = np.arange(probs_test.shape[0])[:, None]
pred_top20[rows, top20_idx] = 1
correct_top20 = (pred_top20 * y_test_dense).sum(axis=1)

cluster_test_df = base_df.iloc[test_idx].copy().reset_index(drop=True)
cluster_test_df["cluster_id"] = test_clusters
cluster_test_df["true_burden"] = true_burden_test
cluster_test_df["correct_top20"] = correct_top20
cluster_test_df["recall_at_20"] = correct_top20 / np.maximum(true_burden_test, 1)

cluster_summary = cluster_test_df.groupby("cluster_id").agg(
    patients=("subject_id", "count"),
    mean_age=("age", "mean"),
    mean_prior_admissions=("number_of_prior_admissions", "mean"),
    mean_true_burden=("true_burden", "mean"),
    mean_correct_top20=("correct_top20", "mean"),
    mean_recall_at_20=("recall_at_20", "mean"),
    mean_recent_los=("recent_number_of_days_stay", "mean"),
).sort_values("patients", ascending=False)
cluster_summary

In [ ]:
def top_terms_per_cluster(labels, y_dense, vocab, top_n=10):
    rows = []
    for c in np.unique(labels):
        mask = labels == c
        counts = y_dense[mask].sum(axis=0)
        top_idx = np.argsort(counts)[::-1][:top_n]
        rows.append({
            "cluster_id": int(c),
            "top_diagnoses": [str(vocab[i]) for i in top_idx],
            "counts": [int(counts[i]) for i in top_idx],
        })
    return rows

cluster_top_dx = top_terms_per_cluster(test_clusters, y_test_dense, diag_vocab, top_n=8)
for row in cluster_top_dx:
    print(f"\nCluster {row['cluster_id']}")
    for name, cnt in zip(row['top_diagnoses'], row['counts']):
        print(f"  - {name} ({cnt})")

In [ ]:
# 2D PCA visualization
pca = PCA(n_components=2, random_state=42)
plot_idx = np.random.default_rng(42).choice(len(all_emb), size=min(12000, len(all_emb)), replace=False)
emb_2d = pca.fit_transform(all_emb[plot_idx])
plot_clusters = all_clusters[plot_idx]

plt.figure(figsize=(9, 7))
plt.scatter(emb_2d[:,0], emb_2d[:,1], c=plot_clusters, s=6, alpha=0.6, cmap="tab10")
plt.title("Patient clusters in learned embedding space (PCA view)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)

## 9) Optional: Fast global threshold tuning

This is optional. For this task, **Top-K is still the main metric**. If you need variable-length outputs, use this faster tuner instead of repeated sklearn calls over the full dense matrix.


In [ ]:
def tune_global_threshold_fast(probs, y_true_dense, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.30, 0.95, 14)
    y_true_bool = (y_true_dense > 0)
    total_true = y_true_bool.sum()
    rows = []
    best = None
    for t in thresholds:
        pred_bool = probs >= t
        tp = np.logical_and(pred_bool, y_true_bool).sum()
        fp = np.logical_and(pred_bool, ~y_true_bool).sum()
        fn = total_true - tp
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        avg_pred = pred_bool.sum(axis=1).mean()
        row = {"t": float(t), "micro_f1": float(f1), "micro_precision": float(precision),
               "micro_recall": float(recall), "avg_pred": float(avg_pred)}
        rows.append(row)
        if best is None or row["micro_f1"] > best["micro_f1"]:
            best = row
    return best, pd.DataFrame(rows)

probs_val, y_val_dense = predict_proba(model, val_loader, device)
best_t, threshold_table = tune_global_threshold_fast(probs_val, y_val_dense)
best_t, threshold_table.head()

## 10) Inspect predictions for a few patients (top‑K)

This helps you see whether predictions are clinically plausible.


In [ ]:
def show_patient_topk(i, probs, y_true_dense, diag_vocab, k=20):
    idx = topk_pred_indices(probs[i:i+1], k=k)[0]
    true = set(np.where(y_true_dense[i] > 0.5)[0].tolist())
    correct = sum((j in true) for j in idx)
    print(f"Row {i}: correct in top-{k} = {correct}/{k} | true labels = {len(true)}")
    for rank, j in enumerate(idx, start=1):
        mark = "✅" if j in true else "  "
        print(f"{mark} {rank:>2}. {diag_vocab[j]}  (p={probs[i, j]:.3f})")

for i in [0, 1, 2]:
    print("="*80)
    show_patient_topk(i, probs_test, y_test_dense, diag_vocab, k=20)
